# NB09 — Mechanistic Flux Validation with MICOM

## The Holy Trinity of Producer Evidence
| Leg | Tool | Question answered |
|---|---|---|
| Statistical | SHAP (NB03) | Which species *correlates* with metabolite abundance in patients? |
| Genomic | GutSMASH (NB06) | Which species *encodes* the biosynthetic gene cluster? |
| **Mechanistic** | **MICOM (NB09)** | **Which species *actually secretes* the metabolite under gut conditions?** |

## Pipeline
```
Species relative abundances (YACHIDA — ALL stages: Healthy, Early_CRC, Advanced_CRC)
  └─ MICOM build()    — community FBA models from AGORA103 database
       └─ tradeoff=0.5 (hardcoded MICOM default; QP optimization requires CPLEX/Gurobi)
            └─ grow() — simulate growth on Western Diet
                 └─ exchanges — metabolite excretion fluxes per species (all metabolites)
                      └─ cross-reference with SHAP producers (NB03) — any metabolite
```

## Requirements
```
pip install micom
```
Download before running:
- **AGORA database:** `agora103_species.qza` from https://zenodo.org/record/7800476
- **Western Diet:** `western_diet_gut.csv` from https://github.com/micom-dev/micom/tree/main/data
- Placed both in `Data/micom/`

In [1]:
import sys, warnings, pickle, re
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

from utils import (
    INTER_DIR, TABLE_DIR, DATA_DIR,
    DATASET_PRIMARY,
    annotate_pathway,
    load_pickle,
)

try:
    import micom
    from micom.workflows import build, grow  # tradeoff not used: hardcoded at 0.5
    print(f"micom {micom.__version__} loaded")
except ImportError:
    raise ImportError(
        "micom is required for this notebook.\n"
        "Install with: pip install micom"
    )

# ── Paths ──────────────────────────────────────────────
MICOM_DIR  = DATA_DIR / "micom"
AGORA_DB   = MICOM_DIR / "agora103_species.qza"
DIET_FILE  = MICOM_DIR / "western_diet_gut.csv"
MODELS_DIR = MICOM_DIR / "models"

# GROWTH_DIR and TRADEOFF_DIR removed: outputs written directly to MICOM_DIR
for d in [MICOM_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Preflight checks
missing = []
if not AGORA_DB.exists():  missing.append(str(AGORA_DB))
if not DIET_FILE.exists(): missing.append(str(DIET_FILE))
if missing:
    print("WARNING — missing required files:")
    for m in missing:
        print(f"  {m}")
    print("Download instructions in notebook header.")
else:
    print("All required files present. Ready to run MICOM.")


micom 0.39.0 loaded
All required files present. Ready to run MICOM.


## Step 1 — Load species relative abundances

In [2]:
pkl     = load_pickle(INTER_DIR / "preprocessed_data.pkl")
meta    = pkl["harmonized_meta"][DATASET_PRIMARY]

# MICOM requires relative (not CLR) abundances.
data_raw_ds = pkl.get("data_raw", {}).get(DATASET_PRIMARY, {})
spe_raw = data_raw_ds.get("species")
if spe_raw is None or (hasattr(spe_raw, "empty") and spe_raw.empty):
    spe_raw = pkl["species_filtered"][DATASET_PRIMARY]
    print("Using species_filtered (no data_raw[species] key found)")

# Include ALL stages (Healthy, Early_CRC, Advanced_CRC) for stage-stratified analysis
spe_all = spe_raw.copy()
print(f"All samples: {len(spe_all)}")
print(f"Stage breakdown:\n{meta['Stage.3Group'].value_counts().to_string()}")
print(f"Species in abundance table: {spe_all.shape[1]}")

# Renormalise each sample to sum = 1 (relative abundance)
spe_all = spe_all.div(spe_all.sum(axis=1), axis=0).fillna(0)

# Keep spe_adv alias — used by the AWRP fallback cell (bf72fe11)
adv_idx = meta[meta["Stage.3Group"] == "Advanced_CRC"].index
spe_adv = spe_all.loc[spe_all.index.isin(adv_idx)]

# ── Species name normalisation ─────────────────────────────────────────────
def extract_binomial(gtdb_str: str) -> tuple:
    """Return (genus, epithet) from a GTDB full taxonomy string.
    Returns (None, None) for GTDB genome-bin IDs with no real species name.
    """
    parts = str(gtdb_str).split(";")
    for p in reversed(parts):
        p = p.strip()
        if p.startswith("s__"):
            name = p[3:].strip()
            tokens = name.split()
            if len(tokens) < 2:
                return (None, None)
            genus, epithet = tokens[0], tokens[1]
            if not (genus[0].isupper() and len(genus) > 1 and genus[1].islower()):
                return (None, None)
            if not epithet[0].islower():
                return (None, None)
            if re.match(r"^sp\d", epithet):
                return (None, None)
            return (genus, epithet)
    return (None, None)

tax_long = (
    spe_all                                              # ← all stages, was spe_adv
    .reset_index()
    .rename(columns={spe_all.index.name or "index": "sample_id"})
    .melt(id_vars="sample_id", var_name="_raw_species", value_name="abundance")
)
tax_long = tax_long[tax_long["abundance"] > 1e-5].copy()
tax_long["sample_id"] = tax_long["sample_id"].astype(str)

parsed = tax_long["_raw_species"].apply(extract_binomial)
tax_long["genus"]   = parsed.apply(lambda x: x[0])
tax_long["species"] = parsed.apply(lambda x: x[1])

n_before = len(tax_long)
tax_long = tax_long.dropna(subset=["genus", "species"]).drop(columns=["_raw_species"])
n_after = len(tax_long)
print(f"Dropped {n_before - n_after:,} rows with unresolvable GTDB genome-bin IDs")

# Add stage label to each row (for downstream stage-stratified aggregation)
_stage_map = {str(k): v for k, v in meta["Stage.3Group"].to_dict().items()}
tax_long["stage"] = tax_long["sample_id"].map(_stage_map)
print(f"Stage distribution in tax_long:\n{tax_long['stage'].value_counts().to_string()}")

n_samples = tax_long["sample_id"].nunique()
n_species = tax_long[["genus", "species"]].drop_duplicates().shape[0]
ex = tax_long[["genus", "species"]].drop_duplicates().head(3)
print(f"\nTaxonomy table: {len(tax_long):,} rows ({n_samples} samples, {n_species} unique genus+species pairs)")
print(f"Example entries:\n{ex.to_string(index=False)}")

Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
All samples: 347
Stage breakdown:
Stage.3Group
Advanced_CRC    163
Healthy         127
Early_CRC        57
Species in abundance table: 57702
Dropped 347,626 rows with unresolvable GTDB genome-bin IDs
Stage distribution in tax_long:
stage
Advanced_CRC    86131
Healthy         64683
Early_CRC       29828

Taxonomy table: 180,642 rows (347 samples, 4086 unique genus+species pairs)
Example entries:
          genus             species
Corynebacterium               durum
        Gemella       haemolysans_B
      Alistipes intestinigallinarum


## Step 2 — Build community FBA models from AGORA103 database

In [3]:
manifest_path = MICOM_DIR / "manifest.csv"

# Expected sample count from tax_long
expected_samples = tax_long["sample_id"].nunique()

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    if len(manifest) == 0:
        print("Cached manifest is empty (likely from a failed build). Deleting and rebuilding ...")
        manifest_path.unlink()
        manifest = None
    elif len(manifest) < expected_samples:
        print(f"Manifest has {len(manifest)} rows but {expected_samples} samples expected.")
        print("Deleting manifest to trigger rebuild (existing model pickles preserved) ...")
        manifest_path.unlink()
        manifest = None
    else:
        print(f"Loading existing manifest ({len(manifest)} samples) ...")
else:
    manifest = None

if manifest is None:
    existing_pickles = len(list(MODELS_DIR.glob("*.pickle")))
    print(f"Building community models — {existing_pickles} existing pickles will be reused ...")
    # Do NOT wipe MODELS_DIR: build() detects existing .pickle files and skips them,
    # so only new sample IDs (Healthy + Early_CRC) will incur build cost.
    manifest = build(
        tax_long,
        out_folder=str(MODELS_DIR),
        model_db=str(AGORA_DB),
        cutoff=1e-4,        # exclude species < 0.01% relative abundance
        threads=4,
    )
    manifest.to_csv(manifest_path, index=False)

print(f"Models built: {len(manifest)}")
if len(manifest) == 0:
    raise RuntimeError(
        "build() returned an empty manifest — no species matched the AGORA database.\n"
        "Run the diagnostic cell (Cell 6) to check species name overlap."
    )

# MICOM 0.39 uses 'found_abundance_fraction'; older versions use 'coverage'
_cov_col = next(
    (c for c in ("coverage", "found_abundance_fraction") if c in manifest.columns),
    None,
)
if _cov_col:
    print(f"Median AGORA abundance coverage: {manifest[_cov_col].median():.1%}")
    low_cov = manifest[manifest[_cov_col] < 0.50]
    if len(low_cov):
        print(f"WARNING: {len(low_cov)} samples have <50% AGORA abundance coverage")
for _stat in ("found_taxa", "total_taxa", "found_fraction"):
    if _stat in manifest.columns:
        print(f"  Median {_stat}: {manifest[_stat].median():.3g}")
print(manifest.head().to_string(index=False))

Loading existing manifest (347 samples) ...
Models built: 347
Median AGORA abundance coverage: 29.4%
  Median found_taxa: 43
  Median total_taxa: 221
  Median found_fraction: 0.197
 sample_id        stage         file  found_taxa  total_taxa  found_fraction  found_abundance_fraction
     10021 Advanced_CRC 10021.pickle        38.0       175.0        0.217143                  0.197738
     10023      Healthy 10023.pickle        41.0       194.0        0.211340                  0.163968
     10025      Healthy 10025.pickle        48.0       222.0        0.216216                  0.172160
     10029      Healthy 10029.pickle        44.0       243.0        0.181070                  0.214844
     10031      Healthy 10031.pickle        58.0       254.0        0.228346                  0.287211


In [4]:
# Diagnostic: check genus+species overlap with AGORA103 database
# Run this if build() reports low coverage — it shows exactly which species matched.
import io
from zipfile import ZipFile

print("=== AGORA Species Name Overlap Diagnostic ===")
user_pairs = set(zip(tax_long["genus"], tax_long["species"]))
print(f"Unique (genus, species) pairs in taxonomy table: {len(user_pairs)}")
print("Sample entries:")
for g, e in list(user_pairs)[:5]:
    print(f"  genus={g!r:30s}  species={e!r}")

if AGORA_DB.exists():
    with ZipFile(str(AGORA_DB), "r") as z:
        manifest_files = [f for f in z.namelist() if f.endswith("manifest.csv")]
        if manifest_files:
            with z.open(manifest_files[0]) as f:
                agora_mdf = pd.read_csv(f)
            agora_pairs = set(zip(agora_mdf["genus"], agora_mdf["species"]))
            overlap = user_pairs & agora_pairs
            pct = 100 * len(overlap) / max(1, len(agora_pairs))
            print(f"\nAGORA103 species: {len(agora_pairs)}")
            print(f"Matched: {len(overlap)} / {len(agora_pairs)} AGORA species ({pct:.1f}%)")
            if overlap:
                print("\nMatched examples (first 10):")
                for g, e in sorted(overlap)[:10]:
                    print(f"  {g} {e}")
            else:
                print("WARNING: Zero overlap! genus+species columns must match AGORA manifest exactly.")
                print("AGORA manifest sample (genus, species):")
                print(agora_mdf[["genus","species"]].head(5).to_string(index=False))
        else:
            print(f"No manifest.csv found inside {AGORA_DB.name}")
else:
    print(f"AGORA DB not found at {AGORA_DB}")
    print("Download from https://zenodo.org/record/7800476 and place in Data/micom/")


=== AGORA Species Name Overlap Diagnostic ===
Unique (genus, species) pairs in taxonomy table: 4086
Sample entries:
  genus='Lacibacter'                    species='cauensis'
  genus='Streptococcus'                 species='pseudopneumoniae'
  genus='Pelethenecus'                  species='faecipullorum'
  genus='Egerieisoma'                   species='faecipullorum'
  genus='Herbivorax'                    species='alkalicellulosi'

AGORA103 species: 587
Matched: 263 / 587 AGORA species (44.8%)

Matched examples (first 10):
  Abiotrophia defectiva
  Acidaminococcus intestini
  Acinetobacter baumannii
  Acinetobacter junii
  Actinomyces graevenitzii
  Actinomyces naeslundii
  Actinomyces urogenitalis
  Actinomyces viscosus
  Adlercreutzia equolifaciens
  Aeromonas caviae


## Step 3 — Find optimal community tradeoff parameter

In [5]:
if len(manifest) == 0:
    raise RuntimeError("Manifest is empty — fix species name matching in Cell 3 first.")

tradeoff_path = MICOM_DIR / "tradeoff_results.csv"

# Load Western Diet medium
medium = pd.read_csv(DIET_FILE)
if "reaction" not in medium.columns or "flux" not in medium.columns:
    raise ValueError(
        f"western_diet_gut.csv must have reaction and flux columns; got: {list(medium.columns)}"
    )
print(f"Western Diet medium: {len(medium)} reactions")
n_converted = medium["reaction"].str.endswith("_e").sum()
medium["reaction"] = medium["reaction"].str.replace(r"_e$", "_m", regex=True)
if n_converted:
    print(f"Converted {n_converted} reaction IDs from _e → _m format")
else:
    print("Reaction IDs already in _m format (western_diet_gut.csv v2+)")

# ── Tradeoff parameter ─────────────────────────────────────────────────────
# The cooperative tradeoff step requires a QP-capable solver (CPLEX/Gurobi).
# community.py patched to prefer osqp > coinor_cbc > hybrid > glpk.
# MICOM default tradeoff: 0.5 for metagenomics, 0.3 for 16S.
# 0.5 used here as data is shotgun metagenomics (Yachida et al., 2019).
optimal_tradeoff = 0.5
print(f"Optimal tradeoff parameter (hardcoded): {optimal_tradeoff:.2f}")
print("(MICOM default for metagenomics: 0.5 — avoids unsolvable QP with GLPK)")

# Write a marker CSV so this cell never re-runs the tradeoff computation.
if not tradeoff_path.exists():
    pd.DataFrame({"tradeoff": [optimal_tradeoff], "note": ["hardcoded"]}).to_csv(
        tradeoff_path, index=False
    )
    print(f"Saved marker: {tradeoff_path.name}")

Western Diet medium: 170 reactions
Converted 170 reaction IDs from _e → _m format
Optimal tradeoff parameter (hardcoded): 0.50
(MICOM default for metagenomics: 0.5 — avoids unsolvable QP with GLPK)


## Step 4 — Simulate community growth on Western Diet

In [6]:
import types as _types

if len(manifest) == 0:
    raise RuntimeError("Manifest is empty — fix species name matching in Step 1 first.")

growth_all_path = MICOM_DIR / "growth_results_all.pkl"
growth_adv_path = MICOM_DIR / "growth_results.pkl"    # legacy: Advanced_CRC only

if growth_all_path.exists():
    print("Loading existing all-stage growth results ...")
    _exchanges_df = pd.read_pickle(growth_all_path)
    growth_results = _types.SimpleNamespace(exchanges=_exchanges_df)
else:
    # ── Reuse existing Advanced_CRC results ───────────────────────────────────────
    adv_exchanges = None
    done_ids = set()
    if growth_adv_path.exists():
        with open(growth_adv_path, "rb") as f:
            gr_adv = pickle.load(f)
        adv_exchanges = gr_adv.exchanges
        done_ids = set(adv_exchanges["sample_id"].astype(str))
        print(f"Loaded Advanced_CRC exchanges: {len(adv_exchanges):,} rows, {len(done_ids)} samples")

    # ── Identify samples still needing grow() ────────────────────────────────────
    all_ids = set(manifest["sample_id"].astype(str))
    missing_ids = all_ids - done_ids
    print(f"Samples needing grow(): {len(missing_ids)} (Healthy + Early_CRC)")

    new_exchanges = None
    _batch_complete = False
    if missing_ids:
        manifest_new = manifest[manifest["sample_id"].astype(str).isin(missing_ids)].copy()

        smoke_path = MICOM_DIR / "growth_smoke_new.pkl"
        if not smoke_path.exists():
            # ── Smoke test: 3 new samples before the full batch ───────────────────
            print("Smoke test: running grow() on 3 new samples (Healthy/Early_CRC) ...")
            gr_smoke = grow(
                manifest_new.sample(n=min(3, len(manifest_new)), random_state=42),
                model_folder=str(MODELS_DIR),
                medium=medium,
                tradeoff=optimal_tradeoff,
                threads=1,
                strategy="none",
                presolve=True,
            )
            with open(smoke_path, "wb") as f:
                pickle.dump(gr_smoke, f)
            print(f"Smoke test PASSED — {len(gr_smoke.exchanges)} exchange rows.")
            print("Next: delete Data/micom/growth_smoke_new.pkl then re-run this cell for full batch.")
            new_exchanges = gr_smoke.exchanges
            _batch_complete = False    # smoke only — do NOT save merged results yet
        else:
            # ── Full batch for remaining samples ─────────────────────────────────
            print(f"Running grow() for {len(manifest_new)} samples (Healthy + Early_CRC) ...")
            gr_new = grow(
                manifest_new,
                model_folder=str(MODELS_DIR),
                medium=medium,
                tradeoff=optimal_tradeoff,
                threads=1,
                strategy="none",
                presolve=True,
            )
            new_exchanges = gr_new.exchanges
            _batch_complete = True

    # ── Merge ─────────────────────────────────────────────────────────────────────
    parts = [p for p in [adv_exchanges, new_exchanges] if p is not None]
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    growth_results = _types.SimpleNamespace(exchanges=combined)

    # ── Save (full batch only — not during smoke test) ────────────────────────────
    if _batch_complete:
        combined.to_pickle(growth_all_path)
        print("Saved growth_results_all.pkl")

print("Growth results ready.")
print(f"Exchanges DataFrame: {growth_results.exchanges.shape}")
print(f"Columns: {list(growth_results.exchanges.columns)}")

Loaded Advanced_CRC exchanges: 315,266 rows, 140 samples
Samples needing grow(): 207 (Healthy + Early_CRC)
Running grow() for 207 samples (Healthy + Early_CRC) ...


Output()

[04/17/26 22:50:00] ERROR    Could not solve cooperative tradeoff for 10066. This can often be fixed by  ]8;id=81104;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=958898;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

[04/17/26 22:57:29] ERROR    Could not solve cooperative tradeoff for 10096. This can often be fixed by  ]8;id=90067;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=64119;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

[04/17/26 23:00:39] ERROR    Could not solve cooperative tradeoff for 10103. This can often be fixed by  ]8;id=843210;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=785160;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

...

[04/18/26 01:46:38] ERROR    Could not solve cooperative tradeoff for 10628. This can often be fixed by  ]8;id=534610;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=535732;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

[04/18/26 02:14:58] ERROR    Could not solve cooperative tradeoff for 10661. This can often be fixed by  ]8;id=441878;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=389646;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

[04/18/26 03:04:06] ERROR    Could not solve cooperative tradeoff for 10761. This can often be fixed by  ]8;id=440008;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=637504;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

[04/18/26 03:05:45] ERROR    Could not solve cooperative tradeoff for 10776. This can often be fixed by  ]8;id=301740;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py\grow.py]8;;\:]8;id=919129;file://C:\Users\andna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\micom\workflows\grow.py#70\70]8;;\
                             enabling `presolve`, choosing more permissive atol and rtol arguments, or             
                             by checking that medium fluxes are > atol.                                            

Saved growth_results_all.pkl
Growth results ready.
Exchanges DataFrame: (702311, 8)
Columns: ['taxon', 'sample_id', 'tolerance', 'reaction', 'flux', 'abundance', 'metabolite', 'direction']


In [7]:
# ── Track B fallback: Abundance-Weighted Reaction Presence (AWRP) ───────────
# Use this cell INSTEAD OF the grow() cell if CBC/OSQP still hangs.
#
# Concept: for each species, extract which metabolites it CAN exchange from AGORA103
# (genomic capacity), then weight by relative abundance per sample.
# This is equivalent to the MIMOSA2 approach and is a validated proxy for
# microbial metabolite production potential in the literature.
#
# INTERPRETATION:
#   "flux" = species relative abundance (NOT an FBA-derived flux value).
#   "direction" = "potential" (AWRP cannot determine import vs export — no FBA).
#   This estimates metabolic CAPACITY, not actual secretion rate under Western Diet.
#   Label figures as "Abundance-Weighted Production Potential" not "Flux".
#
# Activate: set USE_AWRP = True
# Requires: Cell 3 must have been run (extract_binomial defined there)

USE_AWRP = False   # <-- flip to True if grow() still hangs after solver patch

if USE_AWRP:
    import zipfile, io, types as _types
    from cobra.io import load_json_model, read_sbml_model

    print("Running AWRP (solver-free metabolic potential scoring) ...")
    agora_cache = {}   # "genus_species" -> sorted list of met_m IDs

    with zipfile.ZipFile(str(AGORA_DB)) as zf:
        model_files = [f for f in zf.namelist()
                       if f.endswith(".json") or f.endswith(".xml")]
        print(f"AGORA archive: {len(model_files)} model files")
        for fname in model_files:
            with zf.open(fname) as fh:
                try:
                    # Dispatch on format: JSON vs SBML/XML
                    if fname.endswith(".json"):
                        m = load_json_model(io.TextIOWrapper(fh, encoding="utf-8"))
                    else:
                        m = read_sbml_model(fh)
                    key = m.id.lower()
                    # Anchored regex strips _e suffix only at end-of-string
                    # to avoid corrupting IDs with _e mid-string (e.g. ser__L_e)
                    agora_cache[key] = sorted({
                        re.sub(r"_e$", "_m", r.id.replace("EX_", ""))
                        for r in m.exchanges
                    })
                except Exception:
                    pass

    print(f"Loaded {len(agora_cache)} AGORA species models")

    # ── Memory-efficient build: one small DataFrame per species (not one dict per row) ──
    # Avoids accumulating millions of Python dicts before pd.DataFrame() is called.
    chunk_dfs = []
    for sample_id, row in spe_adv.iterrows():
        sample_id_str = str(sample_id)
        for col, abund in row.items():
            if abund < 1e-5:
                continue
            genus, epithet = extract_binomial(col)
            if not genus or not epithet:
                continue
            key = f"{genus}_{epithet}".lower()
            if key not in agora_cache:
                continue
            rxns = agora_cache[key]
            chunk_dfs.append(pd.DataFrame({
                "sample_id":  sample_id_str,
                "taxon":      f"{genus} {epithet}",
                "reaction":   rxns,
                "flux":       float(abund),     # abundance proxy, NOT FBA flux
                "metabolite": rxns,             # already stripped to met_m format
                "direction":  "potential",      # AWRP has no directional FBA info
            }))

    if chunk_dfs:
        awrp_exchanges = pd.concat(chunk_dfs, ignore_index=True)
        del chunk_dfs   # release intermediate memory
    else:
        awrp_exchanges = pd.DataFrame(
            columns=["sample_id", "taxon", "reaction", "flux", "metabolite", "direction"]
        )

    print(f"AWRP exchange table: {awrp_exchanges.shape}")
    n_mets = awrp_exchanges["metabolite"].nunique()
    print(f"Unique metabolites represented: {n_mets}")

    # Wrap in SimpleNamespace mirroring GrowthResults — downstream cells (Steps 5-7) unchanged.
    # Use SimpleNamespace (not namedtuple) so the object is picklable regardless of kernel state.
    growth_results = _types.SimpleNamespace(exchanges=awrp_exchanges)
    awrp_exchanges.to_pickle(MICOM_DIR / "growth_results_awrp.pkl")

    print("AWRP complete. growth_results ready for Steps 5-7.")
    print("NOTE: flux values are abundance proxies. Label figures as \"Production Potential\".")
else:
    print("AWRP disabled (USE_AWRP=False). Using grow() results from the cell above.")
    print("Flip USE_AWRP=True if the grow() cell hangs.")

AWRP disabled (USE_AWRP=False). Using grow() results from the cell above.
Flip USE_AWRP=True if the grow() cell hangs.


## Step 5 — Extract and summarise all metabolite excretion fluxes

Maps AGORA/BiGG exchange IDs to human-readable metabolite names, annotates pathways,
and produces a ranked `flux_summary` table covering *all* metabolites (not just polyamines).
Polyamines are available as a filtered subset via `flux_summary[flux_summary.pathway == 'Polyamine']`.

In [8]:
fluxes = growth_results.exchanges.copy()

# Positive flux = net excretion (production into gut lumen)
prod_fluxes = fluxes[fluxes["flux"] > 0].copy()

# Flag unconstrained exchange fluxes (default COBRA upper bounds: 1000 or 100)
UNCONSTRAINED_THRESHOLD = 99.0
n_unconstrained = (prod_fluxes["flux"] >= UNCONSTRAINED_THRESHOLD).sum()
if n_unconstrained:
    print(f"WARNING: {n_unconstrained} exchange rows at default COBRA bounds "
          f"(flux >= {UNCONSTRAINED_THRESHOLD}) — likely unconstrained. Filtered out.")
    prod_fluxes = prod_fluxes[prod_fluxes["flux"] < UNCONSTRAINED_THRESHOLD].copy()

# ── Add stage label via sample_id → Stage.3Group join ─────────────────────────
_stage_map2 = {str(k): v for k, v in meta["Stage.3Group"].to_dict().items()}
prod_fluxes["stage"] = prod_fluxes["sample_id"].astype(str).map(_stage_map2).fillna("Unknown")
print(f"Stage distribution in prod_fluxes:\n{prod_fluxes['stage'].value_counts().to_string()}")

# ── Strip AGORA/BiGG compartment suffixes ─────────────────────────────────────
_COMPARTMENT_RE = re.compile(r'(?:_[em]|\[e\]|\[m\]|\(e\)|\(m\))$')

def _strip_compartment(met_id):
    return _COMPARTMENT_RE.sub("", str(met_id))

prod_fluxes["met_base"] = prod_fluxes["metabolite"].apply(_strip_compartment)

# ── BiGG ID → human-readable name ─────────────────────────────────────────────
BIGG_NAMES = {
    "ac": "Acetate", "but": "Butyrate", "ppa": "Propionate",
    "for": "Formate", "succ": "Succinate", "fum": "Fumarate",
    "mal_L": "Malate", "lac_L": "Lactate", "lac_D": "D-Lactate",
    "akg": "Alpha-Ketoglutarate", "pyr": "Pyruvate", "cit": "Citrate",
    "oaa": "Oxaloacetate", "oxa": "Oxalate", "acac": "Acetoacetate",
    "acald": "Acetaldehyde", "etoh": "Ethanol",
    "hdca": "Palmitate", "ocdca": "Stearate", "ocdcea": "Oleate",
    "ddca": "Dodecanoate", "ttdca": "Tetradecanoate",
    "pac": "Phenylacetate", "hxa": "Hexanoate",
    "ptrc": "Putrescine", "spmd": "Spermidine", "spmi": "Spermine",
    "agmt": "Agmatine", "cadav": "Cadaverine",
    "orn": "Ornithine", "arg_L": "Arginine",
    "ala_L": "Alanine", "ala_D": "D-Alanine", "gly": "Glycine",
    "val_L": "Valine", "leu_L": "Leucine", "ile_L": "Isoleucine",
    "pro_L": "Proline", "phe_L": "Phenylalanine", "trp_L": "Tryptophan",
    "tyr_L": "Tyrosine", "ser_L": "Serine", "thr_L": "Threonine",
    "cys_L": "Cysteine", "met_L": "Methionine", "asn_L": "Asparagine",
    "asp_L": "Aspartate", "glu_L": "Glutamate", "gln_L": "Glutamine",
    "his_L": "Histidine", "lys_L": "Lysine",
    "gbbtn": "Gamma-Butyrobetaine", "crn": "Carnitine",
    "4abut": "GABA", "urea": "Urea",
    "indole": "Indole",
    "ade": "Adenine", "gua": "Guanine", "xan": "Xanthine",
    "hxan": "Hypoxanthine", "ura": "Uracil",
    "adn": "Adenosine", "gsn": "Guanosine", "uri": "Uridine",
    "ins": "Inosine", "cytd": "Cytidine",
    "amp": "AMP", "nmn": "NMN",
    "dad_2": "Deoxyadenosine", "dcyt": "Deoxycytidine",
    "dgsn": "Deoxyguanosine", "thymd": "Thymidine", "drib": "Deoxyribose",
    "ribflv": "Riboflavin", "fol": "Folate", "thm": "Thiamine",
    "nac": "Nicotinate", "ncam": "Nicotinamide",
    "pydx": "Pyridoxal", "pydxn": "Pyridoxine", "pydam": "Pyridoxamine",
    "pnto_R": "Pantothenate", "thf": "Tetrahydrofolate",
    "glc_D": "Glucose", "fru": "Fructose", "gal": "Galactose",
    "man": "Mannose", "xyl_D": "Xylose", "arab_D": "D-Arabinose",
    "arab_L": "L-Arabinose", "rmn": "Rhamnose", "fuc_L": "Fucose",
    "rib_D": "Ribose", "glcn": "Gluconate", "glcur": "Glucuronate",
    "galur": "Galacturonate", "lcts": "Lactose", "malt": "Maltose",
    "sucr": "Sucrose", "tre": "Trehalose", "melib": "Melibiose",
    "cellb": "Cellobiose",
    "h2": "Hydrogen", "co2": "CO2", "h2s": "Hydrogen Sulfide",
    "o2": "Oxygen", "nh4": "Ammonium", "no2": "Nitrite",
    "h": "Proton", "h2o": "Water",
    "pi": "Phosphate", "so4": "Sulfate",
    "k": "Potassium", "ca2": "Calcium", "mg2": "Magnesium",
    "fe2": "Iron(II)", "fe3": "Iron(III)", "mn2": "Manganese",
    "zn2": "Zinc", "cu2": "Copper", "cobalt2": "Cobalt",
    "cl": "Chloride",
    "tma": "Trimethylamine", "chol": "Choline",
    "cholate": "Cholate", "7dhcdchol": "7-Dehydrocholesterol",
    "pheme": "Heme", "sheme": "Siroheme",
    "glyc": "Glycerol", "glyc3p": "Glycerol-3-Phosphate",
    "glyclt": "Glycolate",
    "g6p": "Glucose-6-Phosphate", "mnl": "Mannitol",
    "gam": "Glucosamine", "acgam": "N-Acetylglucosamine",
    "acnam": "N-Acetylneuraminate", "acmana": "N-Acetylmannosamine",
    "bglc": "Beta-Glucoside", "cgly": "Cysteinylglycine",
    "gthrd": "Glutathione (reduced)", "gthox": "Glutathione (oxidized)",
    "chor": "Chorismate", "4hbz": "4-Hydroxybenzoate", "4abz": "4-Aminobenzoate",
    "actn_R": "R-Acetoin", "lald_L": "L-Lactaldehyde",
}

BIGG_TO_KEGG = {
    "ac": "C00033", "but": "C00246", "ppa": "C00163",
    "for": "C00058", "succ": "C00042", "fum": "C00122",
    "mal_L": "C00149", "lac_L": "C00186", "lac_D": "C00196",
    "akg": "C00026", "pyr": "C00022", "cit": "C00158",
    "oaa": "C00036", "acac": "C00164", "acald": "C00084", "etoh": "C00469",
    "hdca": "C00249", "ocdca": "C01571", "ocdcea": "C00712",
    "ddca": "C02679", "ttdca": "C06424",
    "pac": "C07086",
    "ptrc": "C00134", "spmd": "C00315", "spmi": "C00750",
    "agmt": "C00261", "cadav": "C01672",
    "orn": "C00077", "arg_L": "C00062",
    "ala_L": "C00041", "gly": "C00037", "val_L": "C00183",
    "leu_L": "C00123", "ile_L": "C00407", "pro_L": "C00148",
    "phe_L": "C00079", "trp_L": "C00078", "tyr_L": "C00082",
    "ser_L": "C00065", "thr_L": "C00188", "cys_L": "C00097",
    "met_L": "C00073", "asn_L": "C00152", "asp_L": "C00049",
    "glu_L": "C00025", "gln_L": "C00064", "his_L": "C00135",
    "lys_L": "C00047", "4abut": "C00334", "urea": "C00086",
    "ribflv": "C00255", "fol": "C00504", "thm": "C00378",
    "nac": "C00253", "ncam": "C00153",
    "pydxn": "C00314", "pydx": "C00250", "pydam": "C00534",
    "pnto_R": "C00864",
    "glc_D": "C00031", "fru": "C00095", "gal": "C00124",
    "man": "C00159", "xyl_D": "C00181", "rib_D": "C00121",
    "glcur": "C00191", "lcts": "C00243", "malt": "C00208",
    "sucr": "C00089",
    "ade": "C00147", "gua": "C00242", "xan": "C00385",
    "hxan": "C00262", "ura": "C00106", "adn": "C00212",
    "gsn": "C00387", "uri": "C00299", "ins": "C00294",
    "amp": "C00020", "nmn": "C00455",
    "h2": "C00282", "co2": "C00011", "h2s": "C00283",
    "nh4": "C01342", "no2": "C00088",
    "tma": "C00789", "chol": "C00187", "cholate": "C00695",
    "indole": "C00463", "g6p": "C00092", "glcn": "C00257",
}

prod_fluxes["met_name"] = prod_fluxes["met_base"].map(BIGG_NAMES).fillna(prod_fluxes["met_base"])
prod_fluxes["kegg_id"]  = prod_fluxes["met_base"].map(BIGG_TO_KEGG)
prod_fluxes["pathway"]  = prod_fluxes["kegg_id"].apply(
    lambda k: annotate_pathway(k) if pd.notna(k) else "Other"
)

print(f"Total excretion events (flux>0, filtered): {len(prod_fluxes):,}")
print(f"Unique metabolite base IDs: {prod_fluxes['met_base'].nunique()}")
named_pct = prod_fluxes['met_name'].ne(prod_fluxes['met_base']).mean()
print(f"Named metabolites: {prod_fluxes['met_name'].ne(prod_fluxes['met_base']).sum():,} ({named_pct:.0%})")

# ── Aggregate: mean flux per (metabolite, taxon, stage) ──────────────────────
# Stage dimension is now included so downstream visualisation can compare
# Healthy vs Early_CRC vs Advanced_CRC producers for each metabolite.
flux_summary = (
    prod_fluxes
    .groupby(["met_name", "met_base", "taxon", "pathway", "stage"])["flux"]
    .agg(mean_flux="mean", std_flux="std", n_samples="count")
    .reset_index()
    .sort_values(["stage", "mean_flux"], ascending=[True, False])
)

# ── Reconstruct full binomial names from MICOM epithet-only taxa ──────────────
# MICOM taxon = bare species epithet (community.py sets id = taxonomy["species"]).
# Only reconstruct when the epithet maps to exactly one genus (unambiguous).
if "tax_long" in dir():
    epithet_genera = (
        tax_long[["genus", "species"]]
        .drop_duplicates()
        .groupby("species")["genus"]
        .apply(list)
        .to_dict()
    )
    def _epithet_to_binomial(ep):
        genera = epithet_genera.get(ep, [])
        return f"{genera[0]} {ep}" if len(genera) == 1 else ep
    flux_summary["taxon_full"] = flux_summary["taxon"].apply(_epithet_to_binomial)
else:
    flux_summary["taxon_full"] = flux_summary["taxon"]
    print("WARNING: tax_long not in scope. Run Cell 3 first for full binomial mapping.")

flux_summary.to_csv(TABLE_DIR / "micom_flux_summary_staged.csv", index=False)
print(f"\nSaved: micom_flux_summary_staged.csv ({len(flux_summary)} rows, "
      f"{flux_summary['met_name'].nunique()} metabolites, "
      f"{flux_summary['stage'].nunique()} stages)")

if len(flux_summary) > 0:
    print("\nTop metabolites by mean flux (Advanced_CRC):")
    top_mets = (flux_summary[flux_summary["stage"] == "Advanced_CRC"]
                .groupby("met_name")["mean_flux"]
                .mean().sort_values(ascending=False).head(15))
    print(top_mets.to_string())
    poly_summary = flux_summary[flux_summary["pathway"] == "Polyamine"]
    if len(poly_summary):
        print(f"\nPolyamine producers across all stages: {len(poly_summary)} rows")
        print(poly_summary[["met_name", "taxon", "stage", "mean_flux", "taxon_full"]]
              .sort_values(["met_name", "stage", "mean_flux"], ascending=[True, True, False])
              .head(15).to_string(index=False))

Stage distribution in prod_fluxes:
stage
Advanced_CRC    51255
Healthy         37319
Early_CRC       15452
Total excretion events (flux>0, filtered): 104,026
Unique metabolite base IDs: 172
Named metabolites: 91,247 (88%)

Saved: micom_flux_summary_staged.csv (10432 rows, 172 metabolites, 3 stages)

Top metabolites by mean flux (Advanced_CRC):
met_name
idon_L              98.270600
Indole              72.251090
Guanine             71.643294
Phenylalanine       69.167960
Uridine             65.341646
Deoxyguanosine      65.191216
Hydrogen Sulfide    63.484037
tym                 63.259493
3mop                63.111828
Putrescine          61.501104
Ornithine           60.062468
Glycine             59.467010
Alanine             58.807779
Isoleucine          58.383966
Tryptophan          58.266269

Polyamine producers across all stages: 220 rows
met_name           taxon        stage  mean_flux                  taxon_full
Arginine         cloacae Advanced_CRC  98.850556        Enterobacter 

## Step 6 — Cross-reference MICOM fluxes with SHAP producer candidates

Assembles the Holy Trinity table for each metabolite shared between MICOM and SHAP.

**Expected result — zero confirmed intersection:** This is structurally expected, not a bug.
- MICOM (AGORA103) models ~818 named, cultured species → predicts well-characterised gut commensals
  as mechanistic producers (e.g., *Bifidobacterium*, *Streptococcus*, *Faecalibacterium*).
- SHAP (Yachida NB03) was trained on 57,000+ GTDB species including ~80% uncultured genome bins
  (`sp000436475`, etc.) that have no AGORA103 metabolic model.
- The two lines of evidence are **complementary, not redundant**: MICOM shows *who produces it*
  mechanistically; SHAP shows *whose abundance correlates with its dysregulation* in CRC.

Step 7 visualises both sides as a dual-panel figure per metabolite.

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
from utils import savefig

trinity_df = pd.DataFrame()   # initialise so downstream cells are safe

shap_path = TABLE_DIR / "shap_producer_candidates.csv"
if not shap_path.exists():
    print(f"SHAP results not found at {shap_path}")
    print("Run NB03 to generate shap_producer_candidates.csv first.")
elif len(flux_summary) == 0:
    print("flux_summary is empty — no metabolite excretion data to cross-reference.")
else:
    # Guard: BIGG_TO_KEGG is defined in cell-13. Raises clearly if cell-13 was skipped
    # (e.g. fresh kernel loading flux_summary from CSV directly).
    if "BIGG_TO_KEGG" not in dir():
        raise NameError(
            "BIGG_TO_KEGG not in scope — run cell-13 first "
            "(or restart kernel from Cell 1 and run in order)"
        )

    shap_df = pd.read_csv(shap_path)

    shap_df = shap_df.copy()
    shap_df["met_name"]   = shap_df["target"].str.replace(r"^C\d+_", "", regex=True)
    shap_df["kegg_id"]    = shap_df["target"].str.extract(r"^(C\d+)_", expand=False)
    shap_df["epithet"]    = shap_df["species"].str.split().str[-1]
    shap_df["genus_shap"] = shap_df["species"].str.split().str[0]

    print(f"SHAP file: {len(shap_df)} rows, {shap_df['met_name'].nunique()} unique metabolites")
    print(f"MICOM flux_summary: {flux_summary['met_name'].nunique()} unique metabolites, "
          f"{flux_summary['stage'].nunique()} stages")

    shap_kegg_set = set(shap_df["kegg_id"].dropna().unique())
    # Build name-lookup map once outside loops — was previously rebuilt per metabolite×stage
    shap_name_map = {n.lower(): n for n in shap_df["met_name"].unique()}
    micom_keggs   = {BIGG_TO_KEGG[b] for b in flux_summary["met_base"].unique()
                     if b in BIGG_TO_KEGG}
    overlapping_keggs = micom_keggs & shap_kegg_set
    print(f"\nKEGG-ID overlap: {len(overlapping_keggs)} shared metabolites")
    if overlapping_keggs:
        print(f"  {sorted(overlapping_keggs)}")

    # ── Stage-stratified cross-reference loop ─────────────────────────────────────
    # For each stage and each shared metabolite, build a per-stage trinity comparison.
    # SHAP panel is stage-independent (trained on all samples in NB03).
    trinity_rows  = []
    genus_summary = {}   # met_name → set of confirmed genera (across all stages)

    for stage in ["Healthy", "Early_CRC", "Advanced_CRC"]:
        flux_stage = flux_summary[flux_summary["stage"] == stage].copy()
        if flux_stage.empty:
            print(f"  {stage}: no MICOM data (skipped)")
            continue

        for met_base_id in flux_stage["met_base"].unique():
            bigg_kegg    = BIGG_TO_KEGG.get(met_base_id)
            met_name_row = flux_stage.loc[
                flux_stage["met_base"] == met_base_id, "met_name"
            ].iloc[0]

            # Locate SHAP rows for this metabolite (use pre-built maps)
            if bigg_kegg and bigg_kegg in shap_kegg_set:
                shap_sub = shap_df[shap_df["kegg_id"] == bigg_kegg]
            elif met_name_row.lower() in shap_name_map:
                shap_sub = shap_df[shap_df["met_name"].str.lower() == met_name_row.lower()]
            else:
                continue

            # MICOM top-10 secretors for this metabolite and stage
            micom_top = (
                flux_stage[flux_stage["met_base"] == met_base_id]
                .nlargest(10, "mean_flux")
                [["taxon", "taxon_full", "mean_flux", "pathway"]]
                .rename(columns={"taxon": "epithet"})
            )
            micom_top = micom_top.assign(
                genus_micom=micom_top["taxon_full"].str.split().str[0]
            )

            # Genus-level confirmation: MICOM genera (this stage) ∩ SHAP genera
            micom_genera_met = set(
                flux_stage[flux_stage["met_base"] == met_base_id]["taxon_full"]
                .str.split().str[0].dropna()
            )
            shap_genera_met  = set(shap_sub["genus_shap"].dropna())
            confirmed_genera_met = micom_genera_met & shap_genera_met
            if confirmed_genera_met:
                genus_summary.setdefault(met_name_row, set()).update(confirmed_genera_met)

            comparison = pd.merge(
                micom_top,
                shap_sub[["epithet", "genus_shap", "species",
                           "shap_importance", "model"]].rename(
                    columns={"shap_importance": "mean_shap"}
                ),
                on="epithet",
                how="outer",
            ).assign(metabolite=met_name_row, kegg_id=bigg_kegg or "", stage=stage)

            comparison["display_species"] = (
                comparison["species"].fillna(comparison["taxon_full"])
            )
            comparison["genus_micom"] = comparison["taxon_full"].str.split().str[0]
            comparison["genus_shap"]  = comparison["genus_shap"].fillna(
                comparison["species"].str.split().str[0]
            )
            comparison["genus_confirmed"] = (
                comparison["genus_micom"].isin(confirmed_genera_met) |
                comparison["genus_shap"].isin(confirmed_genera_met)
            )
            trinity_rows.append(comparison)

    if trinity_rows:
        trinity_df = pd.concat(trinity_rows, ignore_index=True)
        trinity_df.to_csv(TABLE_DIR / "micom_shap_trinity_crossref_staged.csv", index=False)

        n_species_confirmed = trinity_df.dropna(subset=["mean_flux", "mean_shap"]).shape[0]
        n_genus_pairs = sum(len(v) for v in genus_summary.values())

        print(f"\nHoly Trinity cross-reference (stage-stratified): {len(trinity_df)} total rows")
        print(f"  Metabolites with BOTH MICOM + SHAP data:  {trinity_df['metabolite'].nunique()}")
        print(f"  Stages covered:                            {sorted(trinity_df['stage'].unique())}")
        print(f"  Species confirmed at species level:        {n_species_confirmed}")
        print(f"  Metabolite-genus pairs confirmed (genus):  {n_genus_pairs}")
        if genus_summary:
            print("\nGenus-level Holy Trinity confirmations:")
            for met, genera in sorted(genus_summary.items()):
                print(f"  {met:25s}: {', '.join(sorted(genera))}")
        if n_species_confirmed == 0:
            print()
            print("NOTE: Zero species-level confirmation (structurally expected).")
            print("  AGORA103 type strains vs GTDB congener species within the same genus.")
            print("  Genus-level confirmation above is the publishable Trinity evidence.")
    else:
        print("\nNo metabolite overlap between MICOM fluxes and SHAP producers.")
        print(f"KEGG-level overlap found: {len(overlapping_keggs)} metabolites.")

SHAP file: 960 rows, 49 unique metabolites
MICOM flux_summary: 172 unique metabolites, 3 stages

KEGG-ID overlap: 11 shared metabolites
  ['C00086', 'C00092', 'C00122', 'C00123', 'C00186', 'C00191', 'C00255', 'C00314', 'C00334', 'C00407', 'C02679']

Holy Trinity cross-reference (stage-stratified): 848 total rows
  Metabolites with BOTH MICOM + SHAP data:  11
  Stages covered:                            ['Advanced_CRC', 'Early_CRC', 'Healthy']
  Species confirmed at species level:        0
  Metabolite-genus pairs confirmed (genus):  13

Genus-level Holy Trinity confirmations:
  GABA                     : Alistipes, Bifidobacterium
  Isoleucine               : Bifidobacterium, Escherichia, Fusobacterium, Klebsiella, Megasphaera
  Lactate                  : Collinsella, Dialister, Streptococcus
  Leucine                  : Escherichia
  Riboflavin               : Intestinibacter
  Urea                     : Alistipes

NOTE: Zero species-level confirmation (structurally expected).
  AGORA

## Step 6b — Stage-Stratified DA × MICOM Enrichment

Cross-references MICOM mechanistic producers with species that are **differentially
abundant** in early or advanced CRC (from NB02). Unlike SHAP predictors (GTDB genome bins),
DA species are typically named, cultured bacteria — making them far more likely to appear
in the AGORA103 model database.

- `★E` — species is also DA-significant in **Early CRC** (Healthy vs Early, p < 0.05)
- `★A` — species is also DA-significant in **Advanced CRC** (Healthy vs Advanced, p < 0.05)

Note: uncorrected p-value threshold used (BH correction over 500+ tests with n=57 has
insufficient power; p < 0.05 is the same threshold applied in `da-micom-code`).

In [10]:
# ── DA × MICOM intersection ───────────────────────────────────────────────────────
# Load differential abundance species from NB02 and flag MICOM producers that
# are also DA-significant in early or advanced CRC.
# Unlike SHAP (GTDB genome bins), DA species are named cultured bacteria →
# more likely to have AGORA103 models → real mechanistic + clinical overlap.

DA_PVAL = 0.05  # uncorrected p-value: BH over 500 tests with n=57 has no power

da_early_path = TABLE_DIR / "da_species_Healthy_vs_Early_CRC.csv"
da_adv_path   = TABLE_DIR / "da_species_Healthy_vs_Advanced_CRC.csv"

_da_flags_ready = False

if da_early_path.exists() and da_adv_path.exists() and not trinity_df.empty:
    da_early_df = pd.read_csv(da_early_path)
    da_adv_df   = pd.read_csv(da_adv_path)

    da_early_df["epithet"] = da_early_df["feature"].str.split().str[-1]
    da_adv_df["epithet"]   = da_adv_df["feature"].str.split().str[-1]

    da_early_sig = set(da_early_df.loc[da_early_df["pval"] < DA_PVAL, "epithet"])
    da_adv_sig   = set(da_adv_df.loc[da_adv_df["pval"] < DA_PVAL, "epithet"])

    print(f"DA Early CRC significant species (p<{DA_PVAL}): {len(da_early_sig)}")
    if da_early_sig:
        print(f"  Examples: {sorted(da_early_sig)[:5]}")
    print(f"DA Advanced CRC significant species (p<{DA_PVAL}): {len(da_adv_sig)}")
    if da_adv_sig:
        print(f"  Examples: {sorted(da_adv_sig)[:5]}")

    # Flag trinity_df (drives Cell 19 visualisation)
    trinity_df["da_early"] = trinity_df["epithet"].isin(da_early_sig)
    trinity_df["da_adv"]   = trinity_df["epithet"].isin(da_adv_sig)

    # Flag flux_summary (for the saved CSV)
    flux_summary["da_early"] = flux_summary["taxon"].isin(da_early_sig)
    flux_summary["da_adv"]   = flux_summary["taxon"].isin(da_adv_sig)

    # Save DA × MICOM overlap table (now includes stage column)
    da_micom_overlap = flux_summary[
        flux_summary["da_early"] | flux_summary["da_adv"]
    ].copy()
    da_micom_overlap.to_csv(TABLE_DIR / "micom_da_overlap_staged.csv", index=False)
    print(f"\nDA × MICOM overlap: {len(da_micom_overlap)} rows, "
          f"{da_micom_overlap['met_name'].nunique()} metabolites, "
          f"{da_micom_overlap['taxon'].nunique()} species")

    if len(da_micom_overlap):
        print("\nMechanistic producers confirmed by CRC differential abundance (top 20):")
        print(da_micom_overlap[
            ["met_name", "taxon_full", "stage", "mean_flux", "pathway", "da_early", "da_adv"]
        ].sort_values("mean_flux", ascending=False).head(20).to_string(index=False))
    else:
        print(f"\nNo DA × MICOM overlap found at p < {DA_PVAL}.")

    _da_flags_ready = True

else:
    reason = []
    if not da_early_path.exists(): reason.append(f"missing {da_early_path.name}")
    if not da_adv_path.exists():   reason.append(f"missing {da_adv_path.name}")
    if trinity_df.empty:           reason.append("trinity_df is empty (run NB03 first to generate SHAP data)")
    print("DA enrichment skipped:", "; ".join(reason) or "unknown reason")
    print("Run NB02 to generate DA results, then re-run this cell.")

# Ensure columns exist so Cell 19 plotting is safe regardless of DA availability
if not _da_flags_ready and not trinity_df.empty:
    trinity_df["da_early"] = False
    trinity_df["da_adv"]   = False

DA Early CRC significant species (p<0.05): 36
  Examples: ['excrementavium', 'faecium', 'flavescens_C', 'intestini', 'intestinigallinarum']
DA Advanced CRC significant species (p<0.05): 43
  Examples: ['bifidum', 'circumdentaria', 'eligens', 'eligens_A', 'faecis']

DA × MICOM overlap: 797 rows, 94 metabolites, 9 species

Mechanistic producers confirmed by CRC differential abundance (top 20):
            met_name                   taxon_full        stage  mean_flux             pathway  da_early  da_adv
       Deoxycytidine        Alistipes onderdonkii Advanced_CRC  97.393946               Other      True   False
             Formate Faecalibacterium prausnitzii Advanced_CRC  95.725187               Other      True   False
             Formate                    intestini Advanced_CRC  94.848031               Other      True   False
            Hydrogen                    intestini    Early_CRC  94.569016               Other      True   False
   4-Hydroxybenzoate Faecalibacterium prausni

## Step 7 — Visualisation: Dual-Evidence Plot (MICOM flux | SHAP importance)

For each shared metabolite, shows the top-5 mechanistic producers (MICOM, blue) alongside
the top-5 statistical predictors (SHAP, orange) as complementary lines of evidence.
Species confirmed by both are highlighted in green (expected to be rare given AGORA103 vs GTDB gap).

In [11]:
if not trinity_df.empty:
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    mets_to_plot = trinity_df["metabolite"].dropna().unique().tolist()
    # Sort by total MICOM producer count across all stages; cap at 12
    mets_to_plot = sorted(
        mets_to_plot,
        key=lambda m: trinity_df[
            (trinity_df["metabolite"] == m) & trinity_df["mean_flux"].notna()
        ].shape[0],
        reverse=True,
    )[:12]

    n_rows = len(mets_to_plot)
    TOP_N   = 5
    MAX_LBL = 22    # truncate long SHAP species names

    # Stage colors matching utils.PALETTE_3GROUP
    COL_H    = "#4CAF50"    # Healthy   — green
    COL_E    = "#FFC107"    # Early_CRC — amber
    COL_A    = "#F44336"    # Advanced_CRC — red
    COL_SHAP = "#E65100"    # SHAP statistical predictors — orange
    COL_BOTH = "#2E7D32"    # genus-confirmed Trinity — dark green
    STAGE_COLS = {"Healthy": COL_H, "Early_CRC": COL_E, "Advanced_CRC": COL_A}
    STAGES = ["Healthy", "Early_CRC", "Advanced_CRC"]

    fig, axes = plt.subplots(
        n_rows, 4,
        figsize=(22, max(5, 3.5 * n_rows)),
        gridspec_kw={"wspace": 0.20, "hspace": 0.55},
    )
    if n_rows == 1:
        axes = axes.reshape(1, 4)

    for row_idx, met in enumerate(mets_to_plot):
        sub = trinity_df[trinity_df["metabolite"] == met]

        # Genera confirmed at genus level for this metabolite (across all stages)
        confirmed_genera = (
            set(sub.loc[sub["genus_confirmed"] == True, "genus_micom"].dropna()) |
            set(sub.loc[sub["genus_confirmed"] == True, "genus_shap"].dropna())
        )

        # ── Track max flux across all three stages for shared x-axis ────────────
        stage_max_flux = 0.0

        # ── MICOM panels (cols 0, 1, 2) — one per stage ─────────────────────────
        for col_idx, stage in enumerate(STAGES):
            ax = axes[row_idx][col_idx]
            col_color = STAGE_COLS[stage]
            sub_stage = sub[sub["stage"] == stage]
            micom_sub = sub_stage[sub_stage["mean_flux"].notna()].nlargest(TOP_N, "mean_flux")

            if len(micom_sub):
                labels = []
                for _, r in micom_sub.iterrows():
                    name = str(r.get("taxon_full") or r.get("epithet") or "").split()[0]
                    if r.get("da_early", False):
                        name += " \u2605E"
                    elif r.get("da_adv", False):
                        name += " \u2605A"
                    labels.append(name)
                colors = [
                    COL_BOTH if r.get("genus_micom") in confirmed_genera else col_color
                    for _, r in micom_sub.iterrows()
                ]
                ax.barh(range(len(micom_sub)), micom_sub["mean_flux"].values,
                        color=colors, edgecolor="white", height=0.65)
                ax.set_yticks(range(len(micom_sub)))
                ax.set_yticklabels(labels, fontsize=7)
                stage_max_flux = max(stage_max_flux, micom_sub["mean_flux"].max())
            else:
                ax.set_yticks([])
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes,
                        ha="center", va="center", fontsize=7, color="grey")

            ax.invert_xaxis()
            ax.invert_yaxis()
            ax.set_xlabel("MICOM flux\n(mmol/gDW/h)", fontsize=6.5, color=col_color)
            ax.tick_params(axis="x", labelsize=6)
            ax.spines[["top", "right"]].set_visible(False)

        # Enforce shared x-axis scale across the 3 MICOM panels
        if stage_max_flux > 0:
            for col_idx in range(3):
                axes[row_idx][col_idx].set_xlim(right=0, left=stage_max_flux * 1.10)

        # ── SHAP panel (col 3) — stage-independent ─────────────────────────────
        ax_s = axes[row_idx][3]
        # SHAP values are stage-independent (NB03 trained on all samples).
        # trinity_df has one SHAP row per stage per species — deduplicate to one row
        # per species. keep="first" is deterministic and safe since mean_shap is
        # identical across stage copies of the same (epithet, metabolite) pair.
        shap_rows = sub[sub["mean_shap"].notna()]
        shap_sub2 = (
            shap_rows
            .drop_duplicates(subset=["epithet", "metabolite"], keep="first")
            .nlargest(TOP_N, "mean_shap")
        )
        if len(shap_sub2):
            labels_s = []
            for _, r in shap_sub2.iterrows():
                full = " ".join(str(r.get("species") or r.get("epithet") or "").split()[:2])
                if len(full) > MAX_LBL:
                    full = full[:MAX_LBL - 1] + "\u2026"
                labels_s.append(full)
            colors_s = [
                COL_BOTH if r.get("genus_shap") in confirmed_genera else COL_SHAP
                for _, r in shap_sub2.iterrows()
            ]
            ax_s.barh(range(len(shap_sub2)), shap_sub2["mean_shap"].values,
                      color=colors_s, edgecolor="white", height=0.65)
            ax_s.set_yticks(range(len(shap_sub2)))
            ax_s.set_yticklabels(labels_s, fontsize=7)
            ax_s.invert_yaxis()
        ax_s.set_xlabel("Mean |SHAP|\n(NB03, all stages)", fontsize=6.5, color=COL_SHAP)
        ax_s.tick_params(axis="x", labelsize=6)
        ax_s.yaxis.set_ticks_position("right")
        ax_s.yaxis.set_label_position("right")
        ax_s.spines[["top", "left"]].set_visible(False)

    # ── Finalise layout before computing axes positions ──────────────────────────
    fig.subplots_adjust(top=0.93, bottom=0.07, left=0.05, right=0.97)

    # ── Column headers above first row (one per panel) ───────────────────────────
    stage_labels = ["Healthy (MICOM)", "Early CRC (MICOM)", "Advanced CRC (MICOM)",
                    "SHAP predictors"]
    stage_colors = [COL_H, COL_E, COL_A, COL_SHAP]
    for ci, (lbl, clr) in enumerate(zip(stage_labels, stage_colors)):
        pos = axes[0][ci].get_position()
        fig.text((pos.x0 + pos.x1) / 2, pos.y1 + 0.018,
                 lbl, ha="center", va="bottom", fontsize=8, color=clr)

    # ── Metabolite names: centered above all 4 panels ────────────────────────────
    for row_idx, met in enumerate(mets_to_plot):
        pos_l = axes[row_idx][0].get_position()    # leftmost (Healthy) panel
        pos_r = axes[row_idx][3].get_position()    # rightmost (SHAP) panel
        center_x = (pos_l.x0 + pos_r.x1) / 2
        label_y  = pos_l.y1 + 0.006
        fig.text(center_x, label_y, met,
                 ha="center", va="bottom",
                 fontsize=9, fontweight="bold", color="#222222")

    # ── Legend ───────────────────────────────────────────────────────────────────
    legend_elements = [
        Patch(facecolor=COL_H,    label="MICOM Healthy"),
        Patch(facecolor=COL_E,    label="MICOM Early CRC"),
        Patch(facecolor=COL_A,    label="MICOM Advanced CRC"),
        Patch(facecolor=COL_SHAP, label="SHAP: statistical predictor"),
        Patch(facecolor=COL_BOTH, label="Confirmed at genus level (Trinity)"),
        Line2D([0], [0], linewidth=0, marker="",
               label="\u2605E / \u2605A = also DA in Early / Advanced CRC",
               color="black"),
    ]
    fig.legend(handles=legend_elements, loc="lower center", ncol=3,
               fontsize=8, bbox_to_anchor=(0.5, 0.0), frameon=False)

    fig.suptitle(
        "Holy Trinity: Mechanistic Producers (MICOM) vs Statistical Predictors (SHAP)\n"
        "Stage-stratified MICOM flux \u2014 Healthy | Early CRC | Advanced CRC \u2014 Western Diet",
        fontweight="bold", fontsize=10, y=0.995,
    )
    savefig(fig, "ml", "nb09_micom_shap_trinity_staged.png")
else:
    print("No cross-reference data to plot. Run Steps 5\u20136 first.")

Saved figure: C:\Users\andna\Documents\KI\Results\figures\ml\nb09_micom_shap_trinity_staged.png


## Step 8 — Community Excretion Heatmap Atlas

Three complementary heatmaps built from the MICOM `grow()` exchange fluxes:

| Figure | Shape | What it shows |
|---|---|---|
| **A** | Sample × Metabolite clustermap | Community metabolic output per sample; rows stage-ordered, cols clustered by co-excretion |
| **B** | Stage × Metabolite summary | Absolute log-flux (top) + Z-score per metabolite (bottom) — highlights stage-specific shifts |
| **C** | Species × Metabolite producer map | Top-30 AGORA species vs all metabolites — who produces what |

**Interpretation note:** `strategy="none"` flux values are LP-feasible but not unique. Treat as presence/absence + relative magnitude, not absolute rates.

In [12]:
# ── Step 8 — Community Excretion Heatmap Atlas ──────────────────────────────────
# Figure A: Sample × Metabolite clustermap  (347 samples × all metabolites)
#            Community flux = Σ species excretion per (sample, metabolite)
#            Rows: stage-ordered, not clustered | Cols: clustered by co-excretion
# Figure B: Stage × Metabolite summary (absolute log-flux + Z-score per metabolite)
# Figure C: Species × Metabolite producer heatmap (top-30 AGORA species)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import numpy as np
from utils import savefig

sns.set_style("white")
plt.rcParams.update({"font.family": "DejaVu Sans"})

if "prod_fluxes" not in dir() or prod_fluxes is None or len(prod_fluxes) == 0:
    raise RuntimeError("prod_fluxes not available — run cell-13 first.")

# ─────────────────────────────────────────────────────────────────────────────────
# 1. Build sample × metabolite community-level matrix
# ─────────────────────────────────────────────────────────────────────────────────
samp_met = (
    prod_fluxes
    .groupby(["sample_id", "met_name"])["flux"]
    .sum()
    .unstack(fill_value=0.0)
)
samp_met.index = samp_met.index.astype(str)

# Prevalence filter: keep metabolites present in ≥10% of samples
_min_prev = max(1, int(0.10 * len(samp_met)))
_met_prev = (samp_met > 0).sum(axis=0)
samp_met  = samp_met.loc[:, _met_prev >= _min_prev]
n_samp, n_met = samp_met.shape
print(f"Community excretion matrix: {n_samp} samples × {n_met} metabolites "
      f"(≥10% prevalence; {(_met_prev < _min_prev).sum()} excluded)")

# ─────────────────────────────────────────────────────────────────────────────────
# 2. Stage ordering and row color bar
# ─────────────────────────────────────────────────────────────────────────────────
_STAGE_ORDER  = ["Healthy", "Early_CRC", "Advanced_CRC"]
_STAGE_COLORS = {"Healthy": "#4CAF50", "Early_CRC": "#FFC107", "Advanced_CRC": "#F44336"}

_stage_lookup = pd.Series({str(k): v for k, v in meta["Stage.3Group"].to_dict().items()})
_stage_ser    = _stage_lookup.reindex(samp_met.index).fillna("Unknown")

_stage_codes = pd.Categorical(
    _stage_ser, categories=_STAGE_ORDER + ["Unknown"], ordered=True
).codes
_sort_idx    = np.argsort(_stage_codes, kind="stable")
_samp_met_s  = samp_met.iloc[_sort_idx].copy()
_stage_s     = _stage_ser.iloc[_sort_idx]

_row_colors = _stage_s.map(_STAGE_COLORS).fillna("#BBBBBB").rename("Stage")

# ─────────────────────────────────────────────────────────────────────────────────
# 3. Metabolite pathway color bar
# ─────────────────────────────────────────────────────────────────────────────────
_met_pw = (
    prod_fluxes[["met_name", "pathway"]]
    .drop_duplicates("met_name")
    .set_index("met_name")["pathway"]
)
_PATHWAY_COLORS = {
    "Polyamine":            "#9C27B0",
    "SCFA":                 "#2196F3",
    "Amino Acid":           "#FF9800",
    "Indole / Tryptophan":  "#009688",
    "Nucleoside":           "#795548",
    "Bile Acid":            "#607D8B",
    "Sugar / Carbohydrate": "#66BB6A",
    "Vitamin":              "#FF80AB",
    "Ion":                  "#90CAF9",
    "Other":                "#EEEEEE",
}
_col_colors_df = pd.DataFrame(
    {"Pathway": _samp_met_s.columns.map(
        lambda m: _PATHWAY_COLORS.get(_met_pw.get(m, "Other"), "#EEEEEE")
    )},
    index=_samp_met_s.columns
)

# ─────────────────────────────────────────────────────────────────────────────────
# 4. DA annotation (optional — requires da-micom-code to have run)
# ─────────────────────────────────────────────────────────────────────────────────
_da_met_early = set()
_da_met_adv   = set()
if ("_da_flags_ready" in dir()) and _da_flags_ready and "da_early" in flux_summary.columns:
    for _m in _samp_met_s.columns:
        _fsub = flux_summary[flux_summary["met_name"] == _m]
        if _fsub["da_early"].any(): _da_met_early.add(_m)
        if _fsub["da_adv"].any():   _da_met_adv.add(_m)
    print(f"DA-flagged metabolites: {len(_da_met_early)} (Early CRC), "
          f"{len(_da_met_adv)} (Advanced CRC)")

# ─────────────────────────────────────────────────────────────────────────────────
# 5. log1p transform
# ─────────────────────────────────────────────────────────────────────────────────
_mat = np.log1p(_samp_met_s)

# ═════════════════════════════════════════════════════════════════════════════════
# FIGURE A — Sample × Metabolite clustermap
# ═════════════════════════════════════════════════════════════════════════════════
_fig_w = max(22, n_met * 0.27)
_fig_h = 18

_cg = sns.clustermap(
    _mat,
    row_cluster=False,           # keep stage order; rows not clustered
    col_cluster=True,            # cluster metabolites by co-excretion
    row_colors=_row_colors,
    col_colors=_col_colors_df,
    cmap="YlOrRd",
    xticklabels=True,
    yticklabels=False,
    linewidths=0,
    figsize=(_fig_w, _fig_h),
    dendrogram_ratio=(0.01, 0.12),
    colors_ratio=(0.012, 0.012),
    cbar_pos=(0.01, 0.87, 0.013, 0.07),
    cbar_kws={"label": "log(1 + flux sum)"},
    vmin=0,
)

# Annotate DA-flagged metabolites on x-axis labels
_xtick_labels = []
for _t in _cg.ax_heatmap.get_xticklabels():
    _txt = _t.get_text()
    _sfx = ("★E" if _txt in _da_met_early else "") + ("★A" if _txt in _da_met_adv else "")
    _xtick_labels.append(_txt + ("  " + _sfx if _sfx else ""))
_cg.ax_heatmap.set_xticklabels(_xtick_labels, rotation=90, fontsize=6.0, ha="right")
_cg.ax_heatmap.set_xlabel("Metabolite (clustered by co-excretion pattern)", fontsize=9)
_cg.ax_heatmap.set_ylabel(f"Samples (n={n_samp}, ordered by stage →)", fontsize=9)

# Stage + pathway legends
_stage_patches = [mpatches.Patch(color=c, label=s) for s, c in _STAGE_COLORS.items()]
_pw_present    = set(_met_pw.reindex(_samp_met_s.columns).dropna().unique())
_pw_patches    = [
    mpatches.Patch(color=c, label=p)
    for p, c in _PATHWAY_COLORS.items()
    if p in _pw_present and p != "Other"
]
_leg1 = _cg.fig.legend(
    handles=_stage_patches, title="CRC Stage",
    loc="lower left", bbox_to_anchor=(0.01, 0.01),
    fontsize=8, title_fontsize=9, frameon=True, borderpad=0.8
)
_cg.fig.legend(
    handles=_pw_patches, title="Metabolite Pathway",
    loc="lower center", bbox_to_anchor=(0.50, 0.01),
    fontsize=7, title_fontsize=8.5, frameon=True, ncol=3
)
_cg.fig.add_artist(_leg1)

_cg.fig.suptitle(
    "MICOM Community Excretion Atlas — All Samples × All Metabolites (Western Diet)\n"
    "Community flux = Σ species excretion | Rows: stage-ordered | Cols: co-excretion clustered\n"
    "★E = DA-significant producer in Early CRC   ★A = DA-significant producer in Advanced CRC",
    fontsize=9, fontweight="bold", y=1.005
)
savefig(_cg.fig, "ml", "nb09_heatmap_sample_metabolite.png")
print(f"Fig A saved: nb09_heatmap_sample_metabolite.png  [{_fig_w:.0f}×{_fig_h} in]")
plt.close(_cg.fig)

# Store clustered column order for use in Figs B and C
if hasattr(_cg, "dendrogram_col") and _cg.dendrogram_col is not None:
    _col_order = _samp_met_s.columns[_cg.dendrogram_col.reordered_ind]
else:
    _col_order = _samp_met_s.columns   # fallback: original order

# ═════════════════════════════════════════════════════════════════════════════════
# FIGURE B — Stage × Metabolite summary (absolute log-flux + Z-score)
# ═════════════════════════════════════════════════════════════════════════════════
_tmp_df = _samp_met_s.copy()
_tmp_df["_stage"] = _stage_s.values
_present_stages = [s for s in _STAGE_ORDER if s in _stage_s.values]
_stage_met = (
    _tmp_df
    .groupby("_stage")[list(_samp_met_s.columns)]
    .mean()
    .reindex(_present_stages)
)[_col_order]

# Z-score per metabolite (column-wise) — highlights relative stage changes
_stage_std = _stage_met.std(axis=0).replace(0, 1)
_stage_z   = (_stage_met - _stage_met.mean(axis=0)) / _stage_std

_fig2, (_ax_abs, _ax_z) = plt.subplots(
    2, 1,
    figsize=(max(20, n_met * 0.27), 6),
    gridspec_kw={"hspace": 0.55}
)
# Top panel: absolute log-flux per stage
sns.heatmap(
    np.log1p(_stage_met), ax=_ax_abs, cmap="YlOrRd",
    linewidths=0.3, linecolor="#E0E0E0",
    xticklabels=False, yticklabels=True, vmin=0,
    cbar_kws={"label": "log(1 + mean flux)", "shrink": 0.55}
)
_ax_abs.set_title("Mean community excretion flux per CRC stage (log scale)", fontsize=9, fontweight="bold")
_ax_abs.set_ylabel("Stage", fontsize=9)
_ax_abs.set_yticklabels(_ax_abs.get_yticklabels(), fontsize=9, rotation=0)

# Bottom panel: Z-scored per metabolite
sns.heatmap(
    _stage_z, ax=_ax_z, cmap="RdBu_r", center=0,
    linewidths=0.3, linecolor="#E0E0E0",
    xticklabels=True, yticklabels=True, vmin=-2.5, vmax=2.5,
    cbar_kws={"label": "Z-score (per metabolite)", "shrink": 0.55}
)
_ax_z.set_xticklabels(_ax_z.get_xticklabels(), rotation=90, fontsize=5.5, ha="right")
_ax_z.set_title(
    "Z-scored excretion per metabolite — positive = elevated in that stage "
    "(same column order as atlas above)", fontsize=9, fontweight="bold"
)
_ax_z.set_ylabel("Stage", fontsize=9)
_ax_z.set_yticklabels(_ax_z.get_yticklabels(), fontsize=9, rotation=0)

_fig2.suptitle("Stage × Metabolite Community Excretion Summary", fontsize=10, fontweight="bold", y=1.02)
savefig(_fig2, "ml", "nb09_heatmap_stage_metabolite.png")
print("Fig B saved: nb09_heatmap_stage_metabolite.png")
plt.close(_fig2)

# ═════════════════════════════════════════════════════════════════════════════════
# FIGURE C — Top-30 Species × All Metabolites producer heatmap
# ═════════════════════════════════════════════════════════════════════════════════
# Top 30 species by summed mean excretion flux across all metabolites and stages
_top_sp = (
    flux_summary
    .groupby("taxon_full")["mean_flux"]
    .sum()
    .sort_values(ascending=False)
    .head(30)
    .index.tolist()
)
_spec_raw = (
    flux_summary[flux_summary["taxon_full"].isin(_top_sp)]
    .groupby(["taxon_full", "met_name"])["mean_flux"]
    .mean()
    .unstack(fill_value=0.0)
)
# Align to Fig A column order; fill metabolites not in flux_summary with 0
_spec_cols = [c for c in _col_order if c in _spec_raw.columns]
_spec_raw  = _spec_raw.reindex(columns=_spec_cols, fill_value=0.0)
# Sort rows (species) by total flux descending
_sp_row_order = _spec_raw.sum(axis=1).sort_values(ascending=False).index
_spec_raw     = _spec_raw.loc[_sp_row_order]

_fig3, _ax3 = plt.subplots(figsize=(max(20, len(_spec_cols) * 0.27), 10))
sns.heatmap(
    np.log1p(_spec_raw),
    ax=_ax3,
    cmap="PuRd",
    linewidths=0,
    xticklabels=True,
    yticklabels=True,
    cbar_kws={"label": "log(1 + mean flux)", "shrink": 0.5}
)
_ax3.set_xticklabels(_ax3.get_xticklabels(), rotation=90, fontsize=6.0, ha="right")
_ax3.set_yticklabels(_ax3.get_yticklabels(), fontsize=7.5, fontstyle="italic", rotation=0)
_ax3.set_xlabel("Metabolite (same column order as atlas)", fontsize=9)
_ax3.set_ylabel("Species (top-30 by total flux)", fontsize=9)
_ax3.set_title(
    "Top-30 Mechanistic Producers × All Metabolites — Mean Excretion Flux (AGORA103, all stages combined)",
    fontsize=10, fontweight="bold"
)
_fig3.tight_layout()
savefig(_fig3, "ml", "nb09_heatmap_species_metabolite.png")
print("Fig C saved: nb09_heatmap_species_metabolite.png")
plt.close(_fig3)

# ─────────────────────────────────────────────────────────────────────────────────
# Summary: most stage-variable metabolites
# ─────────────────────────────────────────────────────────────────────────────────
print(f"\nTop 10 most stage-variable metabolites (std across stage means):")
_met_var = _stage_met.std(axis=0).sort_values(ascending=False).head(10)
for _m, _v in _met_var.items():
    _pw_lbl = _met_pw.get(_m, "?")
    _flag   = ("  ★E" if _m in _da_met_early else "") + ("  ★A" if _m in _da_met_adv else "")
    print(f"  {_m:30s}  [{_pw_lbl:22s}]  stage-std={_v:.2f}{_flag}")


Community excretion matrix: 325 samples × 120 metabolites (≥10% prevalence; 52 excluded)
DA-flagged metabolites: 72 (Early CRC), 67 (Advanced CRC)
Saved figure: C:\Users\andna\Documents\KI\Results\figures\ml\nb09_heatmap_sample_metabolite.png
Fig A saved: nb09_heatmap_sample_metabolite.png  [32×18 in]
Saved figure: C:\Users\andna\Documents\KI\Results\figures\ml\nb09_heatmap_stage_metabolite.png
Fig B saved: nb09_heatmap_stage_metabolite.png
Saved figure: C:\Users\andna\Documents\KI\Results\figures\ml\nb09_heatmap_species_metabolite.png
Fig C saved: nb09_heatmap_species_metabolite.png

Top 10 most stage-variable metabolites (std across stage means):
  Uridine                         [Other                 ]  stage-std=64.50  ★E  ★A
  Acetate                         [SCFA                  ]  stage-std=36.99  ★E  ★A
  Ammonium                        [Other                 ]  stage-std=35.70  ★E  ★A
  Indole                          [Other                 ]  stage-std=32.11  ★E  ★A
  Pheny